# RI-JK UHF Hessian：库伦分解

本文档与 `02-2-decomp_de_J.ipynb` 几乎完全相同，区别仅在于使用 UHF 的密度矩阵。

**UHF 与 RHF 的关键区别**：

- UHF 的密度矩阵按自旋分量存储，记号上 $D_{\mu\nu}^\sigma$（$\sigma \in \{\alpha, \beta\}$），einsum 记号为 `dm0[x, u, v]`（spin 维度记号 `x`）。
- 总密度 $D_{\mu\nu} = D_{\mu\nu}^\alpha + D_{\mu\nu}^\beta$。
- 库伦项 $E_J = \frac{1}{2} (\mu\nu|\kappa\lambda) D_{\mu\nu} D_{\kappa\lambda}$ **只依赖总密度** $D$。
- 因此 J 的所有 skeleton 二阶导数 einsum 表达式与 RHF 形式完全相同，**只需将 RHF 的 `dm0`（即 $2 C_\text{occ} C_\text{occ}^T$）替换为 UHF 的总密度 $D = D^\alpha + D^\beta$**。
- 各子项前的系数（4/2/2、2/2/-2/2、…）来自电子积分的置换对称性，**与自旋无关**，因此保持不变。
- 与 RHF 不同：UHF 中 $D^\sigma$ 各自旋通道占据数为 1（而非 RHF 的 2），所以 $D = D^\alpha + D^\beta$ 不再带 RHF 中的 ×2 因子。

In [1]:
from pyscf import gto, scf, lib
import numpy as np
from functools import partial
from pyscf.df.grad.rhf import _int3c_wrapper

lib.num_threads(16)
np.set_printoptions(5, suppress=True, linewidth=150)
np.einsum = partial(np.einsum, optimize="greedy")

In [2]:
xyz = """
N  0   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""

mol = gto.Mole(atom=xyz, basis="def2-TZVP", charge=2, spin=2, max_memory=32000).build()

In [3]:
mf = scf.UHF(mol).density_fit()
mf.mo_coeff = np.load("nh3_u_hf.npz")["mo_coeff"]
mf.mo_occ = np.load("nh3_u_hf.npz")["mo_occ"]
mf.mo_energy = np.load("nh3_u_hf.npz")["mo_energy"]
mf.with_df.build()
mf.converged = True

In [4]:
mf_hess = mf.Hessian().run()
de_ref = mf_hess.de.copy()
print("de_ref shape:", de_ref.shape)

de_ref shape: (4, 4, 3, 3)


In [5]:
de_J20 = np.load("nh3_u_hf_decomp.npz")["de_J20"]
de_J11 = np.load("nh3_u_hf_decomp.npz")["de_J11"]
de_J02 = np.load("nh3_u_hf_decomp.npz")["de_J02"]

In [6]:
# UHF 索引约定：spin 维度记号 `x`，0 = alpha, 1 = beta
α, β = 0, 1

mo_coeff = mf.mo_coeff           # shape (2, nao, nmo)
mo_occ = mf.mo_occ               # shape (2, nmo)
mo_energy = mf.mo_energy         # shape (2, nmo)
nao = mo_coeff.shape[1]
nmo = mo_coeff.shape[2]

# 按自旋通道分别取占据轨道与密度矩阵 (occ = 1 per spin, no RHF *2)
mocca = mo_coeff[α][:, mo_occ[α] > 0]
moccb = mo_coeff[β][:, mo_occ[β] > 0]

dm0 = np.zeros((2, nao, nao))
dm0[α] = mocca @ mocca.T
dm0[β] = moccb @ moccb.T

# 总密度 dm0_s = dm0[α] + dm0[β]（einsum 中可写作 "xuv -> uv"）
dm0_s = dm0.sum(axis=0)

natm = mol.natm
atmlst = range(natm)
aoslices = mol.aoslice_by_atom()
aux = mf.with_df.auxmol
auxslices = aux.aoslice_by_atom()
naux = aux.nao

# 详细分解

实现原则与 RHF 完全一致（程序越简单越好、不考虑积分对称性、einsum 表达、双重 (A, B) 原子循环、按子项分别核验）。

**针对 UHF 的具体处理**：库伦项只依赖总密度 $D = D^\alpha + D^\beta$，因此本 notebook 中所有 einsum 表达式都直接使用变量 `dm0_s`（即 `dm0.sum(axis=0)`）替代 RHF 的 `dm0`，其余形式与 02-2 一致。

In [7]:
int2c2e = aux.intor("int2c2e")
int2c2e_inv = np.linalg.inv(int2c2e)
int3c2e = _int3c_wrapper(mol, aux, "int3c2e", "s1")()
int3c2e_ip1 = _int3c_wrapper(mol, aux, "int3c2e_ip1", "s1")().reshape([3, nao, nao, naux])
int3c2e_ip2 = _int3c_wrapper(mol, aux, "int3c2e_ip2", "s1")().reshape([3, nao, nao, naux])
int3c2e_ipip1 = _int3c_wrapper(mol, aux, "int3c2e_ipip1", "s1")().reshape([3, 3, nao, nao, naux])
int3c2e_ipvip1 = _int3c_wrapper(mol, aux, "int3c2e_ipvip1", "s1")().reshape([3, 3, nao, nao, naux])
int3c2e_ip1ip2 = _int3c_wrapper(mol, aux, "int3c2e_ip1ip2", "s1")().reshape([3, 3, nao, nao, naux])
int3c2e_ipip2 = _int3c_wrapper(mol, aux, "int3c2e_ipip2", "s1")().reshape([3, 3, nao, nao, naux])
int2c2e_ip1 = aux.intor("int2c2e_ip1")
int2c2e_ipip1 = aux.intor("int2c2e_ipip1").reshape([3, 3, naux, naux])
int2c2e_ip1ip2 = aux.intor("int2c2e_ip1ip2").reshape([3, 3, naux, naux])

### J (basis_2nd)

In [8]:
# (10|0)(0|10)
dbas_J20_1 = np.einsum("tuvP, PQ, sklQ, uv, kl -> tsuk", int3c2e_ip1, int2c2e_inv, int3c2e_ip1, dm0_s, dm0_s)
de_J20_1 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    for B, (_, _, p0B, p1B) in enumerate(aoslices):
        de_J20_1[A, B] += 4 * np.einsum("tsuv -> ts", dbas_J20_1[:, :, p0A:p1A, p0B:p1B])

In [9]:
# (11|0)(0|00)
dbas_J20_2 = np.einsum("tsuvP, PQ, klQ, kl -> tsuv", int3c2e_ipvip1, int2c2e_inv, int3c2e, dm0_s)
de_J20_2 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    for B, (_, _, p0B, p1B) in enumerate(aoslices):
        de_J20_2[A, B] += 2 * np.einsum("tsuv, uv -> ts", dbas_J20_2[:, :, p0A:p1A, p0B:p1B], dm0_s[p0A:p1A, p0B:p1B])

In [10]:
# (20|0)(0|00)
dbas_J20_3 = np.einsum("tsuvP, PQ, klQ, kl -> tsuv", int3c2e_ipip1, int2c2e_inv, int3c2e, dm0_s)
de_J20_3 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    de_J20_3[A, A] += 2 * np.einsum("tsuv, uv -> ts", dbas_J20_3[:, :, p0A:p1A], dm0_s[p0A:p1A])

In [11]:
de_J20_recap = de_J20_1 + de_J20_2 + de_J20_3
assert np.allclose(de_J20_recap, de_J20, atol=1e-5, rtol=1e-4)

### J (basis_1st aux_1st)

In [12]:
# (10|1)(0|0)(0|00)
dbas_J11_1 = np.einsum("tsuvP, PQ, klQ, uv, kl -> tsuP", int3c2e_ip1ip2, int2c2e_inv, int3c2e, dm0_s, dm0_s)
de_J11_1 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_J11_1[A, B] += 2 * np.einsum("tsuP -> ts", dbas_J11_1[:, :, p0A:p1A, p0B:p1B])
de_J11_1 += de_J11_1.transpose(1, 0, 3, 2)

In [13]:
# (10|0)(0|1)(0|00)
dbas_J11_2 = np.einsum("tuvP, PQ, sQR, RS, klS, uv, kl -> tsuR", int3c2e_ip1, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, dm0_s, dm0_s)
de_J11_2 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_J11_2[A, B] += 2 * np.einsum("tsuR -> ts", dbas_J11_2[:, :, p0A:p1A, p0B:p1B])
de_J11_2 += de_J11_2.transpose(1, 0, 3, 2)

In [14]:
# (10|0)(1|0)(0|00)
dbas_J11_3 = np.einsum("tuvP, PQ, sQR, RS, klS, uv, kl -> tsuQ", int3c2e_ip1, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, dm0_s, dm0_s)
de_J11_3 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_J11_3[A, B] += -2 * np.einsum("tsuQ -> ts", dbas_J11_3[:, :, p0A:p1A, p0B:p1B])
de_J11_3 += de_J11_3.transpose(1, 0, 3, 2)

In [15]:
# (10|0)(0|0)(1|00)
dbas_J11_4 = np.einsum("tuvP, PQ, sklQ, uv, kl -> tsuQ", int3c2e_ip1, int2c2e_inv, int3c2e_ip2, dm0_s, dm0_s)
de_J11_4 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_J11_4[A, B] += 2 * np.einsum("tsuQ -> ts", dbas_J11_4[:, :, p0A:p1A, p0B:p1B])
de_J11_4 += de_J11_4.transpose(1, 0, 3, 2)

In [16]:
de_J11_recap = de_J11_1 + de_J11_2 + de_J11_3 + de_J11_4
assert np.allclose(de_J11_recap, de_J11, atol=1e-5, rtol=1e-4)

### J (aux_2nd)

In [17]:
# (00|2)(0|00)
dbas_J02_1 = np.einsum("tsuvP, PQ, klQ, uv, kl -> tsP", int3c2e_ipip2, int2c2e_inv, int3c2e, dm0_s, dm0_s)
de_J02_1 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    de_J02_1[A, A] += np.einsum("tsP -> ts", dbas_J02_1[:, :, p0A:p1A])

In [18]:
# (00|0)(2|0)(0|00)
dbas_J02_2 = np.einsum("uvP, PQ, tsQR, RS, klS, uv, kl -> tsQ", int3c2e, int2c2e_inv, int2c2e_ipip1, int2c2e_inv, int3c2e, dm0_s, dm0_s)
de_J02_2 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    de_J02_2[A, A] += -1 * np.einsum("tsQ -> ts", dbas_J02_2[:, :, p0A:p1A])

In [19]:
# (00|0)(1|1)(0|00)
dbas_J02_3a = np.einsum("uvP, PQ, tsQR, RS, klS, uv, kl -> tsQR", int3c2e, int2c2e_inv, int2c2e_ip1ip2, int2c2e_inv, int3c2e, dm0_s, dm0_s)
de_J02_3a = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_J02_3a[A, B] += -0.5 * np.einsum("tsQR -> ts", dbas_J02_3a[:, :, p0A:p1A, p0B:p1B])
de_J02_3a += de_J02_3a.transpose(1, 0, 3, 2)

In [20]:
# (00|0)(1|0)(0|1)(0|00)
dbas_J02_3b = np.einsum("uvP, PQ, tQR, RS, sST, TU, klU, uv, kl -> tsQT", int3c2e, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, dm0_s, dm0_s)
de_J02_3b = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_J02_3b[A, B] += -0.5 * np.einsum("tsQT -> ts", dbas_J02_3b[:, :, p0A:p1A, p0B:p1B])
de_J02_3b += de_J02_3b.transpose(1, 0, 3, 2)

In [21]:
# (00|1)(1|0)(0|00)
dbas_J02_4 = np.einsum("tuvP, PQ, sQR, RS, klS, uv, kl -> tsPQ", int3c2e_ip2, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, dm0_s, dm0_s)
de_J02_4 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_J02_4[A, B] += -1 * np.einsum("tsPQ -> ts", dbas_J02_4[:, :, p0A:p1A, p0B:p1B])
de_J02_4 += de_J02_4.transpose(1, 0, 3, 2)

In [22]:
# (00|1)(1|00)
dbas_J02_5 = np.einsum("tuvP, PQ, sklQ, uv, kl -> tsPQ", int3c2e_ip2, int2c2e_inv, int3c2e_ip2, dm0_s, dm0_s)
de_J02_5 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_J02_5[A, B] += 0.5 * np.einsum("tsPQ -> ts", dbas_J02_5[:, :, p0A:p1A, p0B:p1B])
de_J02_5 += de_J02_5.transpose(1, 0, 3, 2)

In [23]:
# (00|0)(0|1)(1|0)(0|00)
dbas_J02_6 = np.einsum("uvP, PQ, tRQ, RS, sST, TU, klU, uv, kl -> tsRS", int3c2e, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, dm0_s, dm0_s)
de_J02_6 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_J02_6[A, B] += 0.5 * np.einsum("tsRS -> ts", dbas_J02_6[:, :, p0A:p1A, p0B:p1B])
de_J02_6 += de_J02_6.transpose(1, 0, 3, 2)

In [24]:
# (00|1)(0|1)(0|00)
dbas_J02_7 = np.einsum("tuvP, PQ, sRQ, RS, klS, uv, kl -> tsPR", int3c2e_ip2, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, dm0_s, dm0_s)
de_J02_7 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_J02_7[A, B] += -1 * np.einsum("tsPR -> ts", dbas_J02_7[:, :, p0A:p1A, p0B:p1B])
de_J02_7 += de_J02_7.transpose(1, 0, 3, 2)

In [25]:
# (00|0)(1|0)(1|0)(0|00)
dbas_J02_8 = np.einsum("uvP, PQ, tQR, RS, sST, TU, klU, uv, kl -> tsRT", int3c2e, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, dm0_s, dm0_s)
de_J02_8 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_J02_8[A, B] += 1 * np.einsum("tsRT -> ts", dbas_J02_8[:, :, p0A:p1A, p0B:p1B])
de_J02_8 += de_J02_8.transpose(1, 0, 3, 2)

In [26]:
de_J02_recap = de_J02_1 + de_J02_2 + de_J02_3a + de_J02_3b + de_J02_4 + de_J02_5 + de_J02_6 + de_J02_7 + de_J02_8
assert np.allclose(de_J02_recap, de_J02, atol=1e-5, rtol=1e-4)

## 存储到分解文件

In [27]:
dat = dict(np.load("nh3_u_hf_decomp.npz"))
dat.update({
    # de_J20
    "de_J20_1": de_J20_1,
    "de_J20_2": de_J20_2,
    "de_J20_3": de_J20_3,
    # de_J11
    "de_J11_1": de_J11_1,
    "de_J11_2": de_J11_2,
    "de_J11_3": de_J11_3,
    "de_J11_4": de_J11_4,
    # de_J02
    "de_J02_1": de_J02_1,
    "de_J02_2": de_J02_2,
    "de_J02_3a": de_J02_3a,
    "de_J02_3b": de_J02_3b,
    "de_J02_4": de_J02_4,
    "de_J02_5": de_J02_5,
    "de_J02_6": de_J02_6,
    "de_J02_7": de_J02_7,
    "de_J02_8": de_J02_8,
})
np.savez("nh3_u_hf_decomp.npz", **dat)